# 🦠 COVID-19 Dashboard (Google Colab)
A complete interactive dashboard showing COVID-19 statistics globally and by country, using the [disease.sh](https://disease.sh) live API.

Run each cell below **in order**, top to bottom.

## Step 1 — Install & Import Libraries

In [ ]:
!pip install pandas plotly requests numpy -q

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import requests
from datetime import datetime
import numpy as np
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")

## Step 2 — Data Fetching Functions
These functions pull live COVID-19 data from the disease.sh API.

In [ ]:
def fetch_global_data():
    """Fetch global COVID-19 statistics"""
    try:
        url = "https://disease.sh/v3/covid-19/all"
        response = requests.get(url, timeout=10)
        return response.json()
    except Exception as e:
        print(f"❌ Error fetching global data: {e}")
        return None

def fetch_countries_data():
    """Fetch COVID-19 data for all countries"""
    try:
        url = "https://disease.sh/v3/covid-19/countries"
        response = requests.get(url, timeout=10)
        return response.json()
    except Exception as e:
        print(f"❌ Error fetching countries data: {e}")
        return None

def fetch_historical_data():
    """Fetch historical COVID-19 data for global trends"""
    try:
        url = "https://disease.sh/v3/covid-19/historical/all?lastdays=150"
        response = requests.get(url, timeout=10)
        return response.json()
    except Exception as e:
        print(f"❌ Error fetching historical data: {e}")
        return None

## Step 3 — Load All Data

In [ ]:
print("📊 Loading COVID-19 data...")
global_data = fetch_global_data()
countries_data = fetch_countries_data()
historical_data = fetch_historical_data()

if global_data and countries_data and historical_data:
    print("✅ Data loaded successfully!")
else:
    print("⚠️ Some data could not be loaded")

## Step 4 — Global Overview (Summary Table)

In [ ]:
def display_global_overview():
    """Display global statistics overview"""
    print("\n" + "="*80)
    print("🦠 COVID-19 GLOBAL DASHBOARD - OVERVIEW")
    print("="*80)

    if not global_data:
        print("❌ Global data not available")
        return

    print(f"\n📅 Last Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Create a formatted table
    stats = {
        'Metric': [
            '🔴 Total Cases',
            '⚫ Total Deaths',
            '🟢 Total Recovered',
            '🟠 Active Cases',
            '📊 Critical Cases',
            '🧪 Total Tests'
        ],
        'Count': [
            f"{global_data['cases']:,}",
            f"{global_data['deaths']:,}",
            f"{global_data['recovered']:,}",
            f"{global_data['active']:,}",
            f"{global_data.get('critical', 'N/A'):,}",
            f"{global_data.get('tests', 'N/A'):,}"
        ],
        'Today': [
            f"+{global_data['todayCases']:,}",
            f"+{global_data['todayDeaths']:,}",
            f"+{global_data.get('todayRecovered', 'N/A'):,}",
            f"",
            f"",
            f""
        ]
    }

    df_stats = pd.DataFrame(stats)
    print(df_stats.to_string(index=False))

    # Calculate rates
    print(f"\n📈 KEY RATES:")
    print(f"   • Death Rate: {(global_data['deaths']/global_data['cases']*100):.2f}%")
    print(f"   • Recovery Rate: {(global_data['recovered']/global_data['cases']*100):.2f}%")
    print(f"   • Active Rate: {(global_data['active']/global_data['cases']*100):.2f}%")

# Display overview
display_global_overview()

## Step 5 — Visualization 1: Global Case Distribution (Pie Chart)

In [ ]:
def plot_global_distribution():
    """Create pie chart for global case distribution"""
    if not global_data:
        return

    fig = go.Figure(data=[go.Pie(
        labels=['Recovered', 'Active', 'Deaths'],
        values=[global_data['recovered'], global_data['active'], global_data['deaths']],
        marker=dict(colors=['#2ecc71', '#f39c12', '#e74c3c']),
        hole=0.3,
        hovertemplate='<b>%{label}</b><br>Cases: %{value:,}<br>Percentage: %{percent}<extra></extra>'
    )])

    fig.update_layout(
        title='<b>Global COVID-19 Case Distribution</b>',
        font=dict(size=12),
        height=500,
        showlegend=True
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 1: GLOBAL CASE DISTRIBUTION")
print("="*80)
plot_global_distribution()

## Step 6 — Top 10 Countries Table

In [ ]:
def display_top_countries():
    """Display top 10 countries with most cases"""
    print("\n" + "="*80)
    print("🌍 TOP 10 COUNTRIES BY COVID-19 CASES")
    print("="*80 + "\n")

    if not countries_data:
        print("❌ Countries data not available")
        return

    df = pd.DataFrame(countries_data)
    df['death_rate_%'] = (df['deaths'] / df['cases'] * 100).round(2)
    df['recovery_rate_%'] = (df['recovered'] / df['cases'] * 100).round(2)

    top10 = df.nlargest(10, 'cases')[['country', 'cases', 'deaths', 'recovered', 'death_rate_%', 'recovery_rate_%']]
    top10.columns = ['Country', 'Cases', 'Deaths', 'Recovered', 'Death Rate %', 'Recovery Rate %']
    top10['Cases'] = top10['Cases'].apply(lambda x: f"{x:,}")
    top10['Deaths'] = top10['Deaths'].apply(lambda x: f"{x:,}")
    top10['Recovered'] = top10['Recovered'].apply(lambda x: f"{x:,}")

    print(top10.to_string(index=False))

display_top_countries()

## Step 7 — Visualization 2: Top 15 Countries by Cases (Bar Chart)

In [ ]:
def plot_top_countries():
    """Create bar chart for top countries"""
    if not countries_data:
        return

    df = pd.DataFrame(countries_data)
    top15 = df.nlargest(15, 'cases')

    fig = px.bar(
        top15.sort_values('cases'),
        y='country',
        x='cases',
        title='<b>Top 15 Countries by Total COVID-19 Cases</b>',
        color='cases',
        color_continuous_scale='Reds',
        orientation='h',
        labels={'cases': 'Number of Cases', 'country': 'Country'},
        hover_data={'cases': ':,'}
    )

    fig.update_layout(
        height=600,
        showlegend=False,
        xaxis_title='Number of Cases',
        yaxis_title='Country'
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 2: TOP 15 COUNTRIES BY CASES")
print("="*80)
plot_top_countries()

## Step 8 — Visualization 3: Top 15 Countries by Deaths (Bar Chart)

In [ ]:
def plot_deaths_comparison():
    """Create bar chart comparing deaths"""
    if not countries_data:
        return

    df = pd.DataFrame(countries_data)
    top15_deaths = df.nlargest(15, 'deaths')

    fig = px.bar(
        top15_deaths.sort_values('deaths'),
        y='country',
        x='deaths',
        title='<b>Top 15 Countries by Total Deaths</b>',
        color='deaths',
        color_continuous_scale='Blues',
        orientation='h',
        labels={'deaths': 'Number of Deaths', 'country': 'Country'},
        hover_data={'deaths': ':,'}
    )

    fig.update_layout(
        height=600,
        showlegend=False,
        xaxis_title='Number of Deaths',
        yaxis_title='Country'
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 3: TOP 15 COUNTRIES BY DEATHS")
print("="*80)
plot_deaths_comparison()

## Step 9 — Visualization 4: Death Rate Analysis (Bar Chart)

In [ ]:
def plot_death_rate():
    """Create chart showing death rate by country"""
    if not countries_data:
        return

    df = pd.DataFrame(countries_data)
    df['death_rate'] = (df['deaths'] / df['cases'] * 100).fillna(0)
    df = df[df['death_rate'] > 0].nlargest(20, 'death_rate')

    fig = px.bar(
        df.sort_values('death_rate'),
        y='country',
        x='death_rate',
        title='<b>Top 20 Countries by Death Rate (%)</b>',
        color='death_rate',
        color_continuous_scale='Purples',
        orientation='h',
        labels={'death_rate': 'Death Rate (%)', 'country': 'Country'},
        hover_data={'death_rate': ':.2f'}
    )

    fig.update_layout(
        height=600,
        showlegend=False,
        xaxis_title='Death Rate (%)',
        yaxis_title='Country'
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 4: TOP 20 COUNTRIES BY DEATH RATE")
print("="*80)
plot_death_rate()

## Step 10 — Visualization 5: Global Trends (Last 150 Days)

In [ ]:
def plot_global_trends():
    """Create trend chart for global statistics"""
    if not historical_data:
        return

    # Extract data
    dates = list(historical_data['cases'].keys())
    cases = list(historical_data['cases'].values())
    deaths = list(historical_data['deaths'].values())
    recovered = list(historical_data['recovered'].values())

    # Create figure with secondary y-axis
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=dates, y=cases,
        mode='lines',
        name='Cases',
        line=dict(color='#e74c3c', width=2),
        fill='tozeroy'
    ))

    fig.add_trace(go.Scatter(
        x=dates, y=deaths,
        mode='lines',
        name='Deaths',
        line=dict(color='#c0392b', width=2),
        fill='tozeroy'
    ))

    fig.add_trace(go.Scatter(
        x=dates, y=recovered,
        mode='lines',
        name='Recovered',
        line=dict(color='#2ecc71', width=2),
        fill='tozeroy'
    ))

    fig.update_layout(
        title='<b>Global COVID-19 Trends (Last 150 Days)</b>',
        xaxis_title='Date',
        yaxis_title='Number of Cases',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 5: GLOBAL TRENDS (LAST 150 DAYS)")
print("="*80)
plot_global_trends()

## Step 11 — Visualization 6: Daily New Cases (Bar Chart)

In [ ]:
def plot_daily_growth():
    """Plot daily growth rate"""
    if not historical_data:
        return

    dates = list(historical_data['cases'].keys())
    cases = list(historical_data['cases'].values())

    # Calculate daily growth rate
    daily_cases = [0] + [cases[i] - cases[i-1] for i in range(1, len(cases))]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=dates,
        y=daily_cases,
        name='Daily New Cases',
        marker=dict(color='#e67e22')
    ))

    fig.update_layout(
        title='<b>Daily New COVID-19 Cases (Last 150 Days)</b>',
        xaxis_title='Date',
        yaxis_title='Number of New Cases',
        height=500,
        hovermode='x unified',
        template='plotly_white'
    )

    display(fig)

print("\n" + "="*80)
print("📊 VISUALIZATION 6: DAILY NEW CASES")
print("="*80)
plot_daily_growth()

## Step 12 — Comprehensive Statistics Summary

In [ ]:
def display_statistics_summary():
    """Display comprehensive statistics"""
    print("\n" + "="*80)
    print("📈 COMPREHENSIVE STATISTICS SUMMARY")
    print("="*80)

    if not countries_data:
        return

    df = pd.DataFrame(countries_data)
    df['death_rate'] = (df['deaths'] / df['cases'] * 100).fillna(0)
    df['recovery_rate'] = (df['recovered'] / df['cases'] * 100).fillna(0)

    print(f"\n🌍 GLOBAL STATISTICS:")
    print(f"   • Total Countries/Regions: {len(df)}")
    print(f"   • Total Cases Worldwide: {df['cases'].sum():,}")
    print(f"   • Total Deaths Worldwide: {df['deaths'].sum():,}")
    print(f"   • Total Recovered Worldwide: {df['recovered'].sum():,}")

    print(f"\n📊 AVERAGE STATISTICS PER COUNTRY:")
    print(f"   • Average Cases: {df['cases'].mean():,.0f}")
    print(f"   • Average Deaths: {df['deaths'].mean():,.0f}")
    print(f"   • Average Recovery Rate: {df['recovery_rate'].mean():.2f}%")
    print(f"   • Average Death Rate: {df['death_rate'].mean():.2f}%")

    print(f"\n🔝 EXTREME VALUES:")
    print(f"   • Highest Cases: {df.loc[df['cases'].idxmax(), 'country']} ({df['cases'].max():,})")
    print(f"   • Highest Deaths: {df.loc[df['deaths'].idxmax(), 'country']} ({df['deaths'].max():,})")
    print(f"   • Highest Death Rate: {df.loc[df['death_rate'].idxmax(), 'country']} ({df['death_rate'].max():.2f}%)")
    print(f"   • Highest Recovery Rate: {df.loc[df['recovery_rate'].idxmax(), 'country']} ({df['recovery_rate'].max():.2f}%)")

display_statistics_summary()

## Step 13 — Country Search Function
Look up detailed stats for any single country by name.

In [ ]:
def search_country_stats(country_name):
    """Search and display statistics for a specific country"""
    if not countries_data:
        print("❌ Countries data not available")
        return

    df = pd.DataFrame(countries_data)
    country_data = df[df['country'].str.lower() == country_name.lower()]

    if country_data.empty:
        print(f"❌ Country '{country_name}' not found")
        print("\nAvailable countries:", ', '.join(sorted(df['country'].tolist())[:20]) + "...")
        return

    data = country_data.iloc[0]

    print(f"\n{'='*60}")
    print(f"📍 STATISTICS FOR: {data['country'].upper()}")
    print(f"{'='*60}\n")

    stats = {
        'Metric': [
            '🔴 Total Cases',
            '⚫ Total Deaths',
            '🟢 Total Recovered',
            '🟠 Active Cases',
            '📊 Cases per 1M Population',
            '📊 Deaths per 1M Population',
            '💉 Critical Cases',
            '🧪 Total Tests',
            '📈 Death Rate',
            '✅ Recovery Rate'
        ],
        'Value': [
            f"{data['cases']:,}",
            f"{data['deaths']:,}",
            f"{data['recovered']:,}",
            f"{data['active']:,}",
            f"{data['casesPerOneMillion']:,.2f}",
            f"{data['deathsPerOneMillion']:,.2f}",
            f"{data.get('critical', 'N/A'):,}",
            f"{data.get('tests', 'N/A'):,}",
            f"{(data['deaths']/data['cases']*100):.2f}%" if data['cases'] > 0 else "N/A",
            f"{(data['recovered']/data['cases']*100):.2f}%" if data['cases'] > 0 else "N/A"
        ]
    }

    df_result = pd.DataFrame(stats)
    print(df_result.to_string(index=False))

## Step 14 — Example Searches: India & United States

In [ ]:
# Example: Search for India
print("\n" + "="*80)
print("🔍 EXAMPLE: COUNTRY SEARCH - INDIA")
print("="*80)
search_country_stats("India")

# Example: Search for USA
print("\n" + "="*80)
print("🔍 EXAMPLE: COUNTRY SEARCH - UNITED STATES")
print("="*80)
search_country_stats("United States")

## Step 15 — Final Summary & Usage Instructions

In [ ]:
print("\n" + "="*80)
print("✅ COVID-19 DASHBOARD COMPLETE!")
print("="*80)
print("""
📝 USAGE INSTRUCTIONS:

To search for any country, use:
    search_country_stats("Country Name")

Examples:
    search_country_stats("Brazil")
    search_country_stats("Germany")
    search_country_stats("Japan")
    search_country_stats("Australia")

To get all available countries:
    df = pd.DataFrame(countries_data)
    print(sorted(df['country'].tolist()))
""")
print("📊 Data Source: Disease.sh COVID-19 API")
print(f"🕐 Last Updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80 + "\n")